# Evaluation Node: Ethical and Social
Evaluate `evaluation_ethical_social` outputs against benchmark labels and rerun this node on selected papers.

In [1]:
import os
import sys
import json
from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

ROOT = Path.cwd().resolve()
for _ in range(6):
    if (ROOT / 'nodes').exists():
        break
    if ROOT.parent == ROOT:
        break
    ROOT = ROOT.parent
if not (ROOT / 'nodes').exists():
    raise RuntimeError(f"Could not resolve project root from cwd: {Path.cwd()}")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils.dsrp_state import DSRPState

PILOT = ROOT / 'Pilot_Evaluation'
BENCHMARK_FILE = PILOT / 'DATA_sample_10' / 'Data Science Research Process (DSRP) Framework.csv'
RESULTS_FOLDER = PILOT / 'pilot_study_results'
VECTOR_DB_PATH = PILOT / 'pilot_study_10_vectors'
COLLECTION_NAME = 'pilot_study_10'
EMBEDDING_MODEL = 'text-embedding-3-small'
NODE_KEY = 'evaluation_ethical_social'

DIMENSION_TO_GT = {
    'privacy_protection_reported': 'Privacy_protection_reported',
    'bias_fairness_considered': 'bias_fairness_considered',
    'societal_impact_discussed': 'societal_impact_discussed',
    'ethical_reflection_level': 'Ethical Reflection Level',
}

DIMENSIONS = list(DIMENSION_TO_GT.keys())

In [2]:
def normalize_text(value):
    if value is None or pd.isna(value):
        return None
    text = str(value).strip()
    return text if text else None

def normalize_paper_id(value):
    text = normalize_text(value)
    return text.lower() if text else None

def canonical_yes_no(value):
    text = normalize_text(value)
    if not text:
        return 'No'
    return 'Yes' if text.lower() in {'yes', 'true', '1', 'y'} else 'No'

def canonical_level(value):
    text = normalize_text(value)
    if not text:
        return None
    mapping = {'high': 'High', 'moderate': 'Moderate', 'low': 'Low'}
    return mapping.get(text.lower(), text)

def normalize_dimension(dim, value):
    if dim in {'privacy_protection_reported', 'bias_fairness_considered', 'societal_impact_discussed'}:
        return canonical_yes_no(value)
    if dim == 'ethical_reflection_level':
        return canonical_level(value)
    return value

def load_agent_results(results_folder):
    data = {}
    for paper_dir in sorted(results_folder.glob('*/')):
        results_file = paper_dir / 'aggregated' / 'results.json'
        if not results_file.exists():
            continue
        with open(results_file, 'r', encoding='utf-8') as f:
            parsed = json.load(f)
            pid = normalize_paper_id(parsed.get('paper_id', paper_dir.name))
            if pid:
                data[pid] = parsed.get('dsrp_outputs', {})
    return data

In [3]:
if not BENCHMARK_FILE.exists():
    raise FileNotFoundError(f"Ground truth file not found: {BENCHMARK_FILE}")

csv_mtime = pd.to_datetime(BENCHMARK_FILE.stat().st_mtime, unit='s')
print(f"Reading fresh ground truth from: {BENCHMARK_FILE}")
print(f"CSV last modified: {csv_mtime}")

benchmark_df = pd.read_csv(BENCHMARK_FILE, low_memory=False)
benchmark_df.columns = benchmark_df.columns.str.strip()
required_cols = ['Paper ID', 'Gatekeeper'] + list(DIMENSION_TO_GT.values())
benchmark_eval = benchmark_df[required_cols].copy()
benchmark_eval['Paper ID'] = benchmark_eval['Paper ID'].apply(normalize_paper_id)
benchmark_eval['Gatekeeper'] = benchmark_eval['Gatekeeper'].astype(str).str.strip().str.lower()
for dim, gt_col in DIMENSION_TO_GT.items():
    benchmark_eval[gt_col] = benchmark_eval[gt_col].apply(lambda v: normalize_dimension(dim, v))
benchmark_eval = benchmark_eval[benchmark_eval['Gatekeeper'] == 'include'].copy()

agent_data = load_agent_results(RESULTS_FOLDER)
comparison_rows = []
for _, row in benchmark_eval.iterrows():
    pid = row['Paper ID']
    if pid not in agent_data:
        continue
    node_output = agent_data[pid].get(NODE_KEY, {})
    record = {'Paper ID': pid}
    flags = []
    for dim, gt_col in DIMENSION_TO_GT.items():
        gt = row[gt_col]
        pred = normalize_dimension(dim, node_output.get(dim))
        match = gt == pred
        flags.append(match)
        record[f'GT_{dim}'] = gt
        record[f'Agent_{dim}'] = pred
        record[f'Match_{dim}'] = 'MATCH' if match else 'MISMATCH'
    record['Match_All'] = 'MATCH' if all(flags) else 'MISMATCH'
    comparison_rows.append(record)

comparison_df = pd.DataFrame(comparison_rows)
print(f'Comparable rows: {len(comparison_df)}')
show_cols = ['Paper ID'] + [f'Match_{d}' for d in DIMENSIONS] + ['Match_All']
print(comparison_df[show_cols].to_string(index=False))

Reading fresh ground truth from: C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow\Pilot_Evaluation\DATA_sample_10\Data Science Research Process (DSRP) Framework.csv
CSV last modified: 2026-04-06 02:55:49.646684408
Comparable rows: 7
 Paper ID Match_privacy_protection_reported Match_bias_fairness_considered Match_societal_impact_discussed Match_ethical_reflection_level Match_All
 2018 - 8                             MATCH                          MATCH                           MATCH                          MATCH     MATCH
2019 - 33                             MATCH                          MATCH                           MATCH                          MATCH     MATCH
 2020 - 8                             MATCH                          MATCH                           MATCH                          MATCH     MATCH
2021 - 28                             MATCH                          MATCH                           MATCH                          MATCH     MATCH
2021 

In [4]:
metrics_rows = []
for dim in DIMENSIONS:
    y_true = comparison_df[f'GT_{dim}'].tolist()
    y_pred = comparison_df[f'Agent_{dim}'].tolist()
    metrics_rows.append({
        'dimension': dim,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
    })
print(pd.DataFrame(metrics_rows).to_string(index=False))

                  dimension  accuracy  precision_macro  recall_macro  f1_macro
privacy_protection_reported       1.0              1.0           1.0       1.0
   bias_fairness_considered       1.0              1.0           1.0       1.0
  societal_impact_discussed       1.0              1.0           1.0       1.0
   ethical_reflection_level       1.0              1.0           1.0       1.0


## Iteration Utility

In [5]:
import importlib
import nodes.evaluation_nodes.evaluation_ethical_social as node_module

def _extract_ethical_reasoning(node_output):
    if not isinstance(node_output, dict):
        return '(no reasoning found)', ''
    reasoning = (
        node_output.get('validated_reasoning')
        or node_output.get('reasoning_explanation')
        or node_output.get('reasoning')
        or '(no reasoning found)'
    )
    audit = node_output.get('audit_commentary', '')
    return str(reasoning), str(audit or '')

def _extract_ethical_bibliography(node_output):
    if not isinstance(node_output, dict):
        return []
    bib = node_output.get('validated_bibliography')
    if isinstance(bib, list):
        return bib
    bib = node_output.get('bibliography')
    if isinstance(bib, list):
        return bib
    return []

def run_ethical_iteration(paper_ids, llm_model='gpt-4o-mini', verbose=True):
    importlib.reload(node_module)
    ethical_node = node_module.evaluation_ethical_social_node

    benchmark_lookup = benchmark_eval.set_index('Paper ID')
    rows = []
    raw_outputs = {}
    original_dir = Path.cwd().resolve()
    prev_env_model = os.getenv('DSRP_LLM_MODEL')

    os.chdir(ROOT)
    os.environ['DSRP_LLM_MODEL'] = llm_model

    try:
        for paper_id in paper_ids:
            pid = normalize_paper_id(paper_id)
            if not pid:
                continue

            if pid not in benchmark_lookup.index:
                rows.append({'Paper ID': pid, 'Model': llm_model, 'Error': 'Paper ID not found in benchmark include set'})
                continue

            state = DSRPState(
                paper_id=pid,
                dsrp_outputs={},
                collection_name=COLLECTION_NAME,
                persist_directory=str(VECTOR_DB_PATH),
                embedding_model=EMBEDDING_MODEL,
                gatekeeper={},
            )
            try:
                out = ethical_node(state)
                dsrp_outputs = out.get('dsrp_outputs', state['dsrp_outputs'])
                node_output = dsrp_outputs.get(NODE_KEY, {})
                raw_outputs[pid] = node_output

                record = {'Paper ID': pid, 'Model': llm_model}
                flags = []
                for dim, gt_col in DIMENSION_TO_GT.items():
                    gt = benchmark_lookup.loc[pid, gt_col]
                    pred = normalize_dimension(dim, node_output.get(dim))
                    match = gt == pred
                    flags.append(match)
                    record[f'GT_{dim}'] = gt
                    record[f'New_Agent_{dim}'] = pred
                    record[f'Match_{dim}'] = 'MATCH' if match else 'MISMATCH'
                record['Match_All'] = 'MATCH' if all(flags) else 'MISMATCH'
                rows.append(record)

                if verbose:
                    summary = ' | '.join([f"{d}: {record[f'Match_{d}']}" for d in DIMENSIONS])
                    print(f"{pid}: {summary} | Match_All: {record['Match_All']}")
            except Exception as e:
                rows.append({'Paper ID': pid, 'Model': llm_model, 'Error': str(e)})
                if verbose:
                    print(f'{pid}: ERROR -> {e}')
    finally:
        if prev_env_model is None:
            os.environ.pop('DSRP_LLM_MODEL', None)
        else:
            os.environ['DSRP_LLM_MODEL'] = prev_env_model
        os.chdir(original_dir)

    return pd.DataFrame(rows), raw_outputs

def show_ethical_label_comparison(df):
    if not isinstance(df, pd.DataFrame) or len(df) == 0:
        print('No results to display.')
        return
    cols = ['Paper ID', 'Model']
    for dim in DIMENSIONS:
        cols.extend([f'GT_{dim}', f'New_Agent_{dim}', f'Match_{dim}'])
    if 'Match_All' in df.columns:
        cols.append('Match_All')
    cols = [c for c in cols if c in df.columns]
    print(df[cols].to_string(index=False))

def view_existing_ethical_reasoning(retest_df, retest_outputs, paper_id, show_bibliography=False):
    pid = normalize_paper_id(paper_id)
    rows = retest_df[retest_df['Paper ID'].apply(normalize_paper_id) == pid]
    if len(rows) == 0:
        raise ValueError(f'Paper ID not found in retest_df: {paper_id}')

    row = rows.iloc[-1].to_dict()
    node_output = retest_outputs.get(pid, {})

    print('\n' + '=' * 100)
    print(f"PAPER: {row.get('Paper ID', '(missing)')} | MODEL: {row.get('Model', '(missing)')}")
    print('=' * 100)
    for dim in DIMENSIONS:
        print(f"{dim}")
        print(f"  GT    : {row.get(f'GT_{dim}', '(missing)')}")
        print(f"  Agent : {row.get(f'New_Agent_{dim}', '(missing)')}")
        print(f"  Match : {row.get(f'Match_{dim}', '(missing)')}")
    print(f"\nMatch_All: {row.get('Match_All', '(missing)')}")

    reasoning, audit = _extract_ethical_reasoning(node_output)
    print('\nReasoning:')
    print(reasoning)
    if audit.strip():
        print('\nAudit Commentary:')
        print(audit)

    if show_bibliography:
        bibliography = _extract_ethical_bibliography(node_output)
        print('\nBibliography:')
        if not bibliography:
            print('  (none)')
        else:
            for i, item in enumerate(bibliography, start=1):
                page = item.get('page', '') if isinstance(item, dict) else ''
                section = item.get('section', '') if isinstance(item, dict) else ''
                quote = item.get('direct_quote', '') if isinstance(item, dict) else ''
                print(f"  [{i}] page={page} | section={section}")
                print(f"      quote: {quote}")

def rerun_single_ethical_paper(paper_id, llm_model='gpt-4o-mini', verbose=True):
    df, outputs = run_ethical_iteration([paper_id], llm_model=llm_model, verbose=verbose)
    show_ethical_label_comparison(df)
    return df, outputs

inspect_ethical_reasoning = view_existing_ethical_reasoning

In [6]:
# Re-iterate on all included papers and show per-label comparison
all_included_papers = benchmark_eval['Paper ID'].tolist()
print(f'Total included papers for rerun: {len(all_included_papers)}')

retest_df, retest_outputs = run_ethical_iteration(all_included_papers, llm_model='gpt-4o-mini', verbose=True)

print('\nDetailed label comparison (GT vs Agent):')
show_ethical_label_comparison(retest_df)

summary_cols = ['Paper ID'] + [f'Match_{d}' for d in DIMENSIONS] + ['Match_All']
summary_cols = [c for c in summary_cols if c in retest_df.columns]
print('\nCompact match summary:')
print(retest_df[summary_cols].to_string(index=False))

Total included papers for rerun: 7


C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow\utils\paper_retriever.py:9: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  self._vectorstore = Chroma(


2018 - 8: privacy_protection_reported: MATCH | bias_fairness_considered: MATCH | societal_impact_discussed: MATCH | ethical_reflection_level: MATCH | Match_All: MATCH
2019 - 33: privacy_protection_reported: MATCH | bias_fairness_considered: MATCH | societal_impact_discussed: MATCH | ethical_reflection_level: MATCH | Match_All: MATCH
2020 - 8: privacy_protection_reported: MATCH | bias_fairness_considered: MATCH | societal_impact_discussed: MATCH | ethical_reflection_level: MATCH | Match_All: MATCH
2021 - 28: privacy_protection_reported: MATCH | bias_fairness_considered: MATCH | societal_impact_discussed: MATCH | ethical_reflection_level: MATCH | Match_All: MATCH
2021 - 56: privacy_protection_reported: MATCH | bias_fairness_considered: MATCH | societal_impact_discussed: MATCH | ethical_reflection_level: MATCH | Match_All: MATCH
2023 - 58: privacy_protection_reported: MATCH | bias_fairness_considered: MATCH | societal_impact_discussed: MATCH | ethical_reflection_level: MATCH | Match_All: 

In [7]:
# Helper usage examples

# A) View reasoning from existing retest results (NO rerun)
view_existing_ethical_reasoning(
    retest_df=retest_df,
    retest_outputs=retest_outputs,
    paper_id='2024 - 99',
    show_bibliography=False,
 )




PAPER: 2024 - 99 | MODEL: gpt-4o-mini
privacy_protection_reported
  GT    : No
  Agent : No
  Match : MATCH
bias_fairness_considered
  GT    : No
  Agent : No
  Match : MATCH
societal_impact_discussed
  GT    : No
  Agent : No
  Match : MATCH
ethical_reflection_level
  GT    : Low
  Agent : Low
  Match : MATCH

Match_All: MATCH

Reasoning:
The study does not report any safeguards for privacy, does not discuss any bias or fairness issues, and does not reflect on potential societal consequences. Therefore, all criteria for ethical evaluation are not met.

Audit Commentary:
The ethical classifications are supported by the evidence from the study, confirming that there are significant gaps in privacy protection, bias consideration, and societal impact discussion.


In [ ]:
# B) Re-run on a single paper only (optional)
single_retest_df, single_retest_outputs = rerun_single_ethical_paper(
     paper_id='2024 - 99',
     llm_model='gpt-4o-mini',
     verbose=True,
 )

2024 - 99: privacy_protection_reported: MATCH | bias_fairness_considered: MATCH | societal_impact_discussed: MISMATCH | ethical_reflection_level: MATCH | Match_All: MISMATCH
 Paper ID       Model GT_privacy_protection_reported New_Agent_privacy_protection_reported Match_privacy_protection_reported GT_bias_fairness_considered New_Agent_bias_fairness_considered Match_bias_fairness_considered GT_societal_impact_discussed New_Agent_societal_impact_discussed Match_societal_impact_discussed GT_ethical_reflection_level New_Agent_ethical_reflection_level Match_ethical_reflection_level Match_All
2024 - 99 gpt-4o-mini                             No                                    No                             MATCH                          No                                 No                          MATCH                          Yes                                  No                        MISMATCH                         Low                                Low                          MA